# 🧭 Sequential Boundary Classifiers for Colonoscopy Localization
### VIM Polyp Dataset · Adjacent-Segment Binary Classifiers

**Idea:** Instead of a single multi-class model that tries to classify *all* colon
segments at once, train a **separate binary classifier for each pair of
anatomically adjacent segments** (e.g. `cecum vs ascending`, `ascending vs
transverse`, `transverse vs descending`, ...).

During a real colonoscopy withdrawal, the scope moves through the colon in a
(mostly) fixed anatomical order. A bank of "boundary detectors" — one per
adjacent pair — can each specialize in telling two *neighboring* segments
apart, which is usually an easier problem than telling all segments apart at
once (neighboring segments tend to look more visually similar to each other
than to segments elsewhere in the colon, so a model that only has to solve
that specific pairwise problem can focus its capacity there). Chaining the
boundary detectors together then gives a simple way to estimate roughly
*where the scope is* in the procedure.

**Pipeline Overview:**
1. Environment Setup
2. Configuration — anatomical order of segments (**edit this for your data**)
3. Frame Extraction (same extraction/cleaning logic as the main notebook)
4. Build Adjacent Pairs
5. Per-Pair Dataset Builder (filter → balance → stratified split → YOLO folders)
6. Train One Binary Classifier per Adjacent Pair
7. Evaluate Each Boundary Classifier
8. Summary Dashboard Across All Boundaries
9. (Optional) Chaining Boundary Classifiers into a Position Estimate

> This notebook assumes the same raw video layout as `VIM_YOLO_Localization.ipynb`
> (filenames encoding the anatomical segment at index 5 when split on `-`).
> If your naming convention differs, adjust `extract_segment_from_filename()` in
> Section 3.

---


## 1. Environment Setup

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────
import os
import cv2
import shutil
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {DEVICE}")


## 2. Configuration

**Edit the values in this cell for your dataset before running anything else.**

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────
LOCAL_DATA_DIR = r"D:\Work\Nebras\colonVideosWithLabels"   # same raw video folder as the main notebook

ROOT = Path('vim_polyp_pairwise')
TEMP_FRAMES_DIR = ROOT / 'data' / 'temp_frames'
PAIRWISE_DATA_DIR = ROOT / 'pairwise_data'
RESULTS = ROOT / 'results'
for d in (TEMP_FRAMES_DIR, PAIRWISE_DATA_DIR, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

# Segments to exclude entirely from this experiment (same as the main notebook)
CLASSES_TO_DROP = {'anastomosis', 'rectosigmoid', 'colon'}

# ─── Anatomical order (THE IMPORTANT PART) ───────────────────────────────────
# List your segment class labels in true anatomical withdrawal order, from
# deepest (cecum) to shallowest (rectum). This does NOT need to include every
# label present in the raw data — only labels listed here get a boundary
# classifier trained between them. Anything not listed is simply skipped.
#
# Example matching the request: cecum -> ascending -> transverse
ANATOMICAL_ORDER = [
    'cecum',
    'ascending',
    'transverse',
    'descending',
    'sigmoid',
]

print("Anatomical order configured:")
for i, seg in enumerate(ANATOMICAL_ORDER):
    print(f"  {i}. {seg}")

# ─── Balancing strategy for each pairwise dataset ────────────────────────────
# 'undersample' -> cap both classes in a pair to the smaller class's count
# 'oversample'   -> duplicate the smaller class up to the larger class's count
PAIR_BALANCE_STRATEGY = 'undersample'

# ─── Frame extraction settings ───────────────────────────────────────────────
FRAMES_PER_SECOND = 1  # how many frames per second of video to sample


## 3. Frame Extraction & Cleaning

Same artifact-filtering logic as the main notebook — pitch-black frames and solid color/blue-screen artifacts are skipped.

In [ ]:
def is_unwanted_frame(frame, brightness_thresh=15, variance_thresh=5):
    """Returns True if the frame is pitch black, a solid blue screen, or a blank artifact."""
    means, stddevs = cv2.meanStdDev(frame)
    if max(means) < brightness_thresh:
        return True
    if max(stddevs) < variance_thresh:
        return True
    blue_mean, green_mean, red_mean = means[0][0], means[1][0], means[2][0]
    if blue_mean > 150 and green_mean < 40 and red_mean < 40:
        return True
    return False


def extract_segment_from_filename(filename):
    """
    Extracts the anatomical segment label from a video filename.
    Assumes the VIM Polyp naming convention: segment is token index 5 when
    the filename stem is split on '-'. Adjust this if your filenames differ.
    """
    stem = os.path.splitext(filename)[0]
    parts = stem.split('-')
    if len(parts) > 5:
        return parts[5].lower().strip()
    return None


# ─── Scan raw video directory ────────────────────────────────────────────────
all_video_files = []
if os.path.exists(LOCAL_DATA_DIR):
    for entry in os.scandir(LOCAL_DATA_DIR):
        if not entry.is_file():
            continue
        filename = entry.name
        if filename.startswith('.') or filename.startswith('._'):
            continue
        if not filename.lower().endswith(('.avi', '.mp4', '.mkv')):
            continue

        segment = extract_segment_from_filename(filename)
        if segment is None or segment in CLASSES_TO_DROP:
            continue
        # Only keep videos for segments we actually care about (in ANATOMICAL_ORDER)
        if segment not in ANATOMICAL_ORDER:
            continue

        all_video_files.append((Path(entry.path), segment))

    print(f"✅ Found {len(all_video_files)} relevant videos "
          f"(segments restricted to ANATOMICAL_ORDER).")
else:
    print(f"❌ Local data directory not found at: {LOCAL_DATA_DIR}")

# ─── Extract clean frames ─────────────────────────────────────────────────────
all_pairs = []   # list of (frame_name, segment_label)
skipped_artifacts = 0

if all_video_files:
    for video_path, segment in tqdm(all_video_files, desc="Extracting clean frames"):
        video_id = video_path.stem
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        frame_interval = max(1, int(fps / FRAMES_PER_SECOND))
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                if is_unwanted_frame(frame):
                    skipped_artifacts += 1
                    frame_count += 1
                    continue

                frame_name = f"{video_id}_frame_{frame_count:05d}.jpg"
                frame_path = TEMP_FRAMES_DIR / frame_name
                cv2.imwrite(str(frame_path), frame)
                all_pairs.append((frame_name, segment))

            frame_count += 1
        cap.release()

    print(f"\n✅ Total valid extracted frames: {len(all_pairs)}")
    print(f"🗑️  Skipped {skipped_artifacts} unwanted artifact/blank frames.")

    df_all = pd.DataFrame(all_pairs, columns=['frame_name', 'segment'])
    print("\n📊 Frame counts per segment:")
    print(df_all['segment'].value_counts().reindex(ANATOMICAL_ORDER).to_string())
else:
    print("⚠️  No video files to process — check LOCAL_DATA_DIR and ANATOMICAL_ORDER.")
    df_all = pd.DataFrame(columns=['frame_name', 'segment'])


## 4. Build Adjacent Pairs

Each consecutive pair in `ANATOMICAL_ORDER` becomes one binary classification problem — this is the direct implementation of "a model for cecum vs ascending, another for ascending vs transverse", etc.

In [ ]:
PAIR_LIST = list(zip(ANATOMICAL_ORDER[:-1], ANATOMICAL_ORDER[1:]))

print(f"📋 {len(PAIR_LIST)} adjacent-segment boundary classifiers will be trained:")
for i, (a, b) in enumerate(PAIR_LIST, 1):
    print(f"  {i}. {a}  vs  {b}")


## 5. Per-Pair Dataset Builder

For each pair, this: filters frames to just those two classes, balances the class counts, does a stratified train/val/test split, and lays the files out in the standard YOLO classification folder structure (`train/<class>/*.jpg`, etc.).

In [ ]:
def build_pairwise_dataset(class_a, class_b, df_all, pairwise_root, seed=SEED,
                            balance_strategy=PAIR_BALANCE_STRATEGY,
                            test_size=0.30, val_of_temp=0.50):
    """
    Builds a balanced, stratified, YOLO-classification-format dataset folder
    for a single binary (class_a vs class_b) problem.
    Returns (dataset_dir, counts_dict) or (None, None) if there isn't enough data.
    """
    pair_name = f"{class_a}_vs_{class_b}"
    dataset_dir = pairwise_root / pair_name

    df_pair = df_all[df_all['segment'].isin([class_a, class_b])].copy()
    if df_pair.empty or df_pair['segment'].nunique() < 2:
        print(f"   ⚠️  Skipping {pair_name}: insufficient data for one or both classes.")
        return None, None

    counts_before = df_pair['segment'].value_counts()

    # ─── Balance ──────────────────────────────────────────────────────────
    # NOTE: we intentionally avoid groupby(...).apply(lambda g: g.sample(...))
    # here. In pandas >= 2.2, groupby(col).apply() drops the grouping column
    # ('segment') from the result by default, which caused a downstream
    # KeyError: 'segment' when building the stratified split below. Looping
    # over groups and pd.concat-ing them keeps all original columns intact
    # regardless of pandas version.
    if balance_strategy == 'undersample':
        target_n = int(counts_before.min())
        df_balanced = pd.concat(
            [g.sample(n=target_n, random_state=seed) for _, g in df_pair.groupby('segment')],
            ignore_index=True,
        )
    elif balance_strategy == 'oversample':
        target_n = int(counts_before.max())
        df_balanced = pd.concat(
            [g.sample(n=target_n, replace=True, random_state=seed) for _, g in df_pair.groupby('segment')],
            ignore_index=True,
        )
    else:
        target_n = None
        df_balanced = df_pair.reset_index(drop=True)

    df_balanced = df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    # De-duplicate destination filenames for any oversampled repeats
    df_balanced['dup_idx'] = df_balanced.groupby('frame_name').cumcount()
    df_balanced['dest_frame_name'] = df_balanced.apply(
        lambda r: r['frame_name'] if r['dup_idx'] == 0
        else f"{Path(r['frame_name']).stem}_dup{r['dup_idx']}{Path(r['frame_name']).suffix}",
        axis=1
    )

    # ─── Stratified split ────────────────────────────────────────────────
    src_dest_pairs = list(zip(df_balanced['frame_name'], df_balanced['dest_frame_name']))
    labels = df_balanced['segment'].tolist()

    train_sd, tmp_sd, train_labels, tmp_labels = train_test_split(
        src_dest_pairs, labels, test_size=test_size, stratify=labels, random_state=seed)
    val_sd, test_sd, val_labels, test_labels = train_test_split(
        tmp_sd, tmp_labels, test_size=val_of_temp, stratify=tmp_labels, random_state=seed)

    splits = {
        'train': list(zip(train_sd, train_labels)),
        'val':   list(zip(val_sd,   val_labels)),
        'test':  list(zip(test_sd,  test_labels)),
    }

    # ─── Organize into YOLO classification folders ───────────────────────
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    for split_name, split_data in splits.items():
        for (src_name, dest_name), label in split_data:
            dest_dir = dataset_dir / split_name / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            src_path = TEMP_FRAMES_DIR / src_name
            dest_path = dest_dir / dest_name
            if src_path.exists():
                shutil.copy2(str(src_path), str(dest_path))

    counts = {split_name: pd.Series([l for _, l in split_data]).value_counts().to_dict()
              for split_name, split_data in splits.items()}

    print(f"   ✅ {pair_name}: "
          f"train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])} "
          f"(balance='{balance_strategy}', target/class={target_n})")

    return dataset_dir, counts


print("✅ build_pairwise_dataset() ready.")


## 6. Train One Binary Classifier per Adjacent Pair

Each boundary gets its own lightweight YOLOv8n-cls binary classifier. Training config is intentionally simple (no attention modules) since this is a much easier 2-class problem — swap in `AttentionYOLOClassification` / `AttentionClassificationTrainer` from `VIM_YOLO_Localization.ipynb` here later if a given boundary turns out to need it.

In [ ]:
TRAIN_CFG_BINARY = dict(
    epochs        = 50,
    imgsz         = 224,
    batch         = 16,
    device        = DEVICE,
    optimizer     = 'AdamW',
    lr0           = 1e-3,
    lrf           = 0.01,
    warmup_epochs = 3,
    cos_lr        = True,
    fliplr        = 0.5,
    verbose       = True,
    project       = str(RESULTS),
)

pairwise_models = {}     # pair_name -> trained YOLO object
pairwise_dataset_dirs = {}  # pair_name -> dataset dir
pairwise_counts = {}     # pair_name -> split counts dict

for class_a, class_b in PAIR_LIST:
    pair_name = f"{class_a}_vs_{class_b}"
    print("=" * 60)
    print(f"TRAINING BOUNDARY CLASSIFIER — {pair_name}")
    print("=" * 60)

    dataset_dir, counts = build_pairwise_dataset(class_a, class_b, df_all, PAIRWISE_DATA_DIR)
    if dataset_dir is None:
        continue

    pairwise_dataset_dirs[pair_name] = dataset_dir
    pairwise_counts[pair_name] = counts

    model = YOLO('yolov8n-cls.pt')
    model.train(
        data = str(dataset_dir),
        name = pair_name,
        **TRAIN_CFG_BINARY,
    )
    pairwise_models[pair_name] = model
    print(f"✅ {pair_name} training complete.\n")

print(f"\n🎉 Trained {len(pairwise_models)} / {len(PAIR_LIST)} boundary classifiers.")


## 7. Evaluate Each Boundary Classifier

In [ ]:
def evaluate_pairwise_model(model, dataset_dir, pair_name):
    val_results = model.val(split='test', verbose=False, data=str(dataset_dir))
    top1 = round(float(val_results.metrics.top1), 4)
    return {'Pair': pair_name, 'Top-1 Accuracy': top1}

pairwise_eval_rows = []
for pair_name, model in pairwise_models.items():
    dataset_dir = pairwise_dataset_dirs[pair_name]
    try:
        pairwise_eval_rows.append(evaluate_pairwise_model(model, dataset_dir, pair_name))
        print(f"✅ {pair_name}: Top-1 = {pairwise_eval_rows[-1]['Top-1 Accuracy']:.3f}")
    except Exception as e:
        print(f"⚠️  Evaluation failed for {pair_name}: {e}")

df_pairwise_results = pd.DataFrame(pairwise_eval_rows)
df_pairwise_results.to_csv(RESULTS / 'pairwise_boundary_results.csv', index=False)
print("\n" + df_pairwise_results.to_string(index=False))


### Confusion matrix per boundary

Since each pair is a 2-class problem, this shows exactly which direction of confusion (e.g. mistaking ascending-colon frames for transverse, or vice versa) is more common for each boundary — the frames closest to the true anatomical transition are naturally the hardest for the model.

In [ ]:
def pairwise_confusion_matrix(model, dataset_dir, class_a, class_b, n_samples=150):
    label_to_id = {class_a: 0, class_b: 1}
    y_true, y_pred = [], []

    test_dir = dataset_dir / 'test'
    candidates = []
    for cls_name in (class_a, class_b):
        cls_dir = test_dir / cls_name
        if cls_dir.exists():
            candidates.extend([(p, cls_name) for p in cls_dir.glob('*.jpg')])

    if not candidates:
        print(f"⚠️  No test images found for {class_a}_vs_{class_b}.")
        return

    sample = random.sample(candidates, min(n_samples, len(candidates)))
    for img_path, true_label in tqdm(sample, desc=f"{class_a}_vs_{class_b} eval", leave=False):
        res = model.predict(str(img_path), verbose=False)[0]
        if res.probs is None:
            continue
        pred_name = res.names[int(res.probs.top1)]
        if pred_name not in label_to_id:
            continue
        y_true.append(label_to_id[true_label])
        y_pred.append(label_to_id[pred_name])

    if not y_true:
        print(f"⚠️  No valid predictions for {class_a}_vs_{class_b}.")
        return

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[class_a, class_b])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{class_a} vs {class_b}', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS / f'confusion_{class_a}_vs_{class_b}.png', dpi=150)
    plt.show()


for class_a, class_b in PAIR_LIST:
    pair_name = f"{class_a}_vs_{class_b}"
    if pair_name in pairwise_models:
        pairwise_confusion_matrix(pairwise_models[pair_name], pairwise_dataset_dirs[pair_name],
                                   class_a, class_b)


## 8. Summary Dashboard Across All Boundaries

In [ ]:
if not df_pairwise_results.empty:
    fig, ax = plt.subplots(figsize=(max(6, 2 * len(df_pairwise_results)), 5))
    bars = ax.bar(df_pairwise_results['Pair'], df_pairwise_results['Top-1 Accuracy'],
                  color='#4C72B0', edgecolor='white')
    for b, v in zip(bars, df_pairwise_results['Top-1 Accuracy']):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_ylabel('Top-1 Accuracy')
    ax.set_title('Boundary Classifier Accuracy — Adjacent Segment Pairs', fontweight='bold')
    plt.xticks(rotation=20, ha='right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS / 'pairwise_summary_dashboard.png', dpi=150)
    plt.show()
    print(f"✅ Dashboard saved to {RESULTS / 'pairwise_summary_dashboard.png'}")
else:
    print('⚠️  No results to plot — check that at least one boundary classifier trained successfully.')


## 9. (Optional) Chaining Boundary Classifiers into a Position Estimate

This is a simple heuristic for turning the bank of pairwise models into a rough
"where is the scope right now" estimate for a single frame. It is **not** a
substitute for a proper temporal model (an LSTM/transformer over the video
sequence, or simple smoothing across consecutive frames, would be far more
robust) — but it's a reasonable starting point and a good sanity check that
the boundary classifiers behave the way you'd expect anatomically.

The idea: starting from a current position estimate (or the first segment if
unknown), only ever ask the classifier for the *boundary the scope is
currently near* whether the frame looks like "still here" or "moved to the
next segment". This mirrors how withdrawal actually happens — monotonically
through the anatomical order — rather than asking every boundary classifier
to vote independently, which would just be a more roundabout multi-class
classifier.

In [ ]:
def localize_frame_sequential(image_path, current_idx, pairwise_models, anatomical_order,
                               confidence_threshold=0.6):
    """
    Given a frame and a current position estimate (index into anatomical_order),
    checks the boundary classifier between the current segment and the next one.
    If it's confident the frame belongs to the next segment, advances the position.
    Returns (new_idx, predicted_label, confidence).
    """
    if current_idx >= len(anatomical_order) - 1:
        # already at the last known segment — nothing further to check
        return current_idx, anatomical_order[current_idx], None

    class_a = anatomical_order[current_idx]
    class_b = anatomical_order[current_idx + 1]
    pair_name = f"{class_a}_vs_{class_b}"

    model = pairwise_models.get(pair_name)
    if model is None:
        return current_idx, anatomical_order[current_idx], None

    res = model.predict(str(image_path), imgsz=224, verbose=False)[0]
    pred_idx = int(res.probs.top1)
    pred_label = res.names[pred_idx]
    confidence = float(res.probs.top1conf)

    if pred_label == class_b and confidence >= confidence_threshold:
        return current_idx + 1, pred_label, confidence
    return current_idx, class_a, confidence


# ─── Demo: walk a handful of test-set frames through the chain ──────────────
if pairwise_models:
    demo_idx = 0
    demo_pair = PAIR_LIST[0]
    demo_dir = pairwise_dataset_dirs.get(f"{demo_pair[0]}_vs_{demo_pair[1]}")
    if demo_dir is not None:
        demo_frames = list((demo_dir / 'test' / demo_pair[0]).glob('*.jpg'))[:5]
        print(f"Starting position estimate: {ANATOMICAL_ORDER[demo_idx]}\n")
        for frame_path in demo_frames:
            demo_idx, label, conf = localize_frame_sequential(
                frame_path, demo_idx, pairwise_models, ANATOMICAL_ORDER)
            conf_str = f"{conf:.2%}" if conf is not None else "n/a"
            print(f"{frame_path.name}: estimated segment = {label} (confidence={conf_str})")
else:
    print('⚠️  No trained boundary models available for the demo.')


## 10. Artifacts Checklist

In [ ]:
artifacts = [(RESULTS / 'pairwise_boundary_results.csv', '📋 Per-boundary accuracy table')]
artifacts += [(RESULTS / f'confusion_{a}_vs_{b}.png', f'🎯 Confusion matrix — {a} vs {b}')
              for a, b in PAIR_LIST]
artifacts.append((RESULTS / 'pairwise_summary_dashboard.png', '📊 Summary dashboard'))

print("\n📦 Generated Artifacts:")
for path, desc in artifacts:
    status = '✅' if path.exists() else '⏳'
    print(f"  {status} {desc:50s} {path}")

print(f"\n🎉 Pairwise boundary classifier experiment complete — "
      f"{len(pairwise_models)} model(s) trained.")
